# ПРОДЕТИ_РИСК — PoC прогноза ПОД (2–12 лет)

Proof-of-concept прогностической модели **послеоперационного делирия (ПОД)** по ТЗ.

**Данные:** `PRODETI_RISK_2-12_for_IT.csv` (экспорт из `данные.numbers`)  
**Целевая переменная:** `outcome_ed_calc` (0 = нет ПОД, 1 = ПОД)  
**Модели:** логистическая регрессия, Random Forest, XGBoost


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
ROOT = Path('.').resolve()
DATA_PATH = ROOT / 'PRODETI_RISK_2-12_for_IT.csv'
TARGET = 'outcome_ed_calc'
FEATURES = [
    'age_years', 'sex_clean', 'mypas_pct', 'induction_behavior', 'duration_min',
    'surgery_type_clean_code', 'planned_duration_cat', 'asa_class', 'opioid_used',
]


## 1. Загрузка, фильтрация и предобработка

**Критерии включения** (из протокола исследования, `выводы.docx`):
- возраст **2–12 лет**
- ASA **≤ III** (исключение ASA IV+)
- удаление дубликатов


In [ ]:
df = pd.read_csv('PRODETI_RISK_2-12_for_IT.csv')
print(f'Исходно: n={len(df)}, ПОД={df[TARGET].sum():.0f} ({df[TARGET].mean():.1%})')

# Фильтрация по критериям исследования
clean = df.drop_duplicates(keep='first')
clean = clean[(clean.age_years >= 2) & (clean.age_years <= 12)]
clean = clean[clean.asa_class <= 3].reset_index(drop=True)
clean.to_csv('PRODETI_RISK_2-12_filtered.csv', index=False)
print(f'После фильтрации: n={len(clean)}, ПОД={clean[TARGET].sum():.0f} ({clean[TARGET].mean():.1%})')
clean.head()


In [ ]:
X = clean[FEATURES].apply(pd.to_numeric, errors='coerce')
y = clean[TARGET].astype(int)
print(f'n={len(y)}, ПОД={y.sum()} ({y.mean():.1%})')


## 2. Модели


In [ ]:
models = {
    'Логистическая регрессия': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
}
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_estimators=100)),
    ])
except Exception as e:
    print('XGBoost недоступен:', e)


## 3. Оценка: 5-fold CV и hold-out 80/20


In [ ]:
def metrics(y_true, proba):
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        'AUC': roc_auc_score(y_true, proba),
        'Accuracy': accuracy_score(y_true, pred),
        'Sensitivity': recall_score(y_true, pred),
        'Specificity': tn / (tn + fp),
        'F1': f1_score(y_true, pred),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in models.items():
    proba = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    row = metrics(y, proba)
    row['Model'] = name
    cv_rows.append(row)
pd.DataFrame(cv_rows).set_index('Model').round(3)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
hold_rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    row = metrics(y_test, proba)
    row['Model'] = name
    hold_rows.append(row)
pd.DataFrame(hold_rows).set_index('Model').round(3)


## 4. ROC-кривые


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CV
ax = axes[0]
for name, pipe in models.items():
    proba = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    auc = roc_auc_score(y, proba)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_title('ROC — 5-fold CV')
ax.legend(fontsize=8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')

# Hold-out
ax = axes[1]
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_title('ROC — hold-out 20%')
ax.legend(fontsize=8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
plt.tight_layout()
plt.savefig('reports/roc_cv.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Feature importance (лучшая модель по CV — логистическая регрессия: |коэффициенты|)


In [ ]:
best = models['Логистическая регрессия']
best.fit(X, y)
coef = np.abs(best.named_steps['clf'].coef_[0])
order = np.argsort(coef)[::-1]
labels = [FEATURES[i] for i in order]
plt.figure(figsize=(8,5))
plt.barh(labels[::-1], coef[order][::-1], color='#4C78A8')
plt.xlabel('|Стандартизованный коэффициент|')
plt.title('Логистическая регрессия — важность признаков')
plt.tight_layout()
plt.savefig('reports/feature_importance_best.png', dpi=150, bbox_inches='tight')
plt.show()

for i in order[:5]:
    print(f'{FEATURES[i]}: {coef[i]:.3f}')


## 6. Краткий вывод

- **Когорта после фильтрации:** n=57 (возраст 2–12, без дубликатов), ПОД 57,9%.
- **Лучшая модель по 5-fold CV:** логистическая регрессия (AUC ≈ 0,77).
- **Ключевые предикторы:** длительность анестезии, поведение при индукции — согласуются с `выводы.docx` (см. `summary.md`).

Полный отчёт: `python scripts/train_and_report.py` → `reports/model_metrics.json`.
